# EDW REST client (`RESTesri.edw`) — usage examples

Walks through every public function in `RESTesri/edw.py` against the live USFS EDW ArcGIS REST services:

- `search_services` — find services by keyword/theme
- `get_service_info` — service-level metadata (layers, spatial ref)
- `get_layer_info` — layer-level metadata (fields, capabilities)
- `get_layer_metadata` — FGDC/ISO metadata (abstract, field definitions, domains)
- `query_features` — attribute + spatial feature queries
- `query_features_with_pagination` — queries beyond the 2000-record server cap
- `query_features_analytic` — SQL window functions (RANK, SUM, LAG, ...)
- `top_n_per_group` — convenience wrapper for "top N per group" queries

In [24]:
from RESTesri.edw import (
    search_services,
    get_service_info,
    get_layer_info,
    get_layer_metadata,
    query_features,
    query_features_with_pagination,
    query_features_analytic,
    top_n_per_group,
)

## 1. `search_services` — find services by keyword or theme

Matches on service name, theme description, and a keyword-alias table (e.g. "riparian" pulls in inland-waters/hydro services even though the word never appears in a service name).

In [25]:
# Plain keyword search
fire_services = search_services("fire")
for s in fire_services[:5]:
    print(s["name"], "-", s["theme"])

EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01 - inland_waters
EDW_AerialFireRetardantAvoidanceAreas_Terrestrial_01 - environment
EDW_BurnedAreaEmergencyResponse_01 - environment
EDW_CommWildfireDefenseGrant_01 - environment
EDW_FireOccurrence6thEdition_01 - environment


In [26]:
# Keyword-alias expansion: "riparian" isn't in any service name, but resolves
# to inland_waters/hydro/watershed/aquatic services via _KEYWORD_ALIASES
search_services("riparian")

[{'name': 'EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_AerialFireRetardantAvoidanceAreas_Aquatic_01/MapServer',
  'theme': 'inland_waters',
  'description': 'This data depicts aquatic aerial fire retardant avoidance areas delivered as part of the 2011 Nationwide Aerial Application of Fire Retardant on…'},
 {'name': 'EDW_AquaticOrganismPassage_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_AquaticOrganismPassage_01/MapServer',
  'theme': 'environment',
  'description': 'This dataset provides USFS watershed improvement activities to barriers to upstream migration.'},
 {'name': 'EDW_ExperimentalForestandRange_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_ExperimentalForestandRange_01/MapServer',
  'theme': 'boundaries',
  'description': 'This polygon feature class contains the boundaries of 86 of 87 experimental 

In [27]:
# Filter by theme only (no keyword) — every transportation-themed service
search_services(theme="transportation")

[{'name': 'EDW_MVUM_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MVUM_01/MapServer',
  'theme': 'transportation',
  'description': 'The feature class indicates the specific types of motorized vehicles allowed on the designated routes and their seasons of use.'},
 {'name': 'EDW_MVUM_02',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MVUM_02/MapServer',
  'theme': 'transportation',
  'description': 'A map service on the www depicting Forest Service roads and trails that are designated for motor vehicle use under the official U.S.'},
 {'name': 'EDW_RoadBasic_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_RoadBasic_01/MapServer',
  'theme': 'transportation',
  'description': 'Existing Forest Service roads with attributes representing their characteristics.'},
 {'name': 'EDW_TrailNFSPublish_01',
  'type': 'MapServer',
  'url': 'https://apps.fs.usda.gov/arcx/rest/ser

## 2. `get_service_info` — service-level metadata

Returns description, spatial reference, extent, and the list of layers in a MapServer.

In [55]:
service_name = "EDW_MTBS_01"
info = get_service_info(service_name)

print("Service description (truncated at 200 characters):")
print(info["description"][:200])
print("# of layers in service:", len(info["layers"]))

print("Info on first 5 layers:")
for lyr in info["layers"][:5]:
    print(lyr["id"], lyr["name"])

Service description (truncated at 200 characters):
A map service on the www that depicts Fire Occurrence Locations and Burned Area Boundaries from the beginning of the Landsat Thematic Mapper archive to the present. The Monitoring Trends in Burn Sever
# of layers in service: 84
Info on first 5 layers:
83 2024 Fire Occurrence Locations
82 2024 Burned Area Boundaries
81 2023 Fire Occurrence Locations
80 2023 Burned Area Boundaries
78 2022 Fire Occurrence Locations


## 3. `get_layer_info` — layer-level metadata

Fields, geometry type, and which operations/analytics the layer supports (`capabilities`, `advancedQueryCapabilities`).

In [29]:
layer_id = 63  # "Burned Area Boundaries (All Years)"
layer_info = get_layer_info(service_name, layer_id)
print(layer_info["name"], layer_info["geometryType"])
print("capabilities:", layer_info["capabilities"])
[f["name"] for f in layer_info["fields"]][:10]

Burned Area Boundaries (All Years) esriGeometryPolygon
capabilities: ['Map', 'Query', 'Data']


['objectid',
 'fire_id',
 'fire_name',
 'year',
 'startmonth',
 'startday',
 'fire_type',
 'acres',
 'irwinid',
 'map_id']

## 4. `get_layer_metadata` — FGDC/ISO metadata document

A separate document from `get_layer_info` — carries the dataset abstract/purpose and per-field *definitions* (what a field actually means), plus optional coded-value/range domains.

In [30]:
meta = get_layer_metadata(service_name, layer_id, include_domains=True)
print(meta["title"])
print(meta["abstract"][:300])
print("keywords:", meta["keywords"][:5])

# Field definitions (only some fields are documented by the data provider)
for attr in meta["attributes"][:5]:
    print(attr["name"], "->", attr["definition"] or "(undocumented)")

MTBS_Burn_Area_Boundary
The Monitoring Trends in Burn Severity (MTBS) Program assesses the frequency, extent, and magnitude (size and severity) of all large wildland fires (including wildfires and prescribed fires) in the conterminous United States (CONUS), Alaska, Hawaii, and Puerto Rico from the beginning of the Landsat 
keywords: ['Burn severity', 'Burned area', 'Differenced normalized burn ratio', 'Fire location', 'Fire occurrence']
OBJECTID -> Internal feature number.
FIRE_ID -> (undocumented)
FIRE_NAME -> (undocumented)
YEAR -> (undocumented)
STARTMONTH -> (undocumented)


## 5. `query_features` — attribute and spatial queries

Returns a GeoJSON `FeatureCollection`. `where` filters by attributes; `geometry`/`geometry_type`/`spatial_rel` add a spatial filter.

In [31]:
# Attribute-only query: a single named fire
cameron_peak = query_features(
    service_name, layer_id,
    where="fire_name = 'CAMERON PEAK'",
    out_fields="fire_name,ig_date,acres",
)
cameron_peak["features"]

[{'type': 'Feature',
  'geometry': {'type': 'MultiPolygon',
   'coordinates': [[[[-105.57165247500677, 40.74112106871535],
      [-105.57161636399778, 40.74111551271723],
      [-105.5715933639916, 40.74110851271801],
      [-105.57158736398985, 40.74110551271804],
      [-105.57158536398885, 40.74110151271763],
      [-105.57158736398897, 40.74109851271709],
      [-105.57159336398982, 40.74109451271612],
      [-105.5715753639853, 40.74109151271706],
      [-105.57128836391367, 40.74104751273217],
      [-105.57100536384341, 40.74100651274737],
      [-105.57073436377661, 40.740971512762535],
      [-105.57047736371378, 40.740942512777394],
      [-105.57023736365555, 40.74091851279172],
      [-105.57001436360196, 40.74090051280562],
      [-105.56981136355371, 40.74088851281885],
      [-105.56962636351034, 40.740882512831554],
      [-105.56946236347258, 40.74088251284351],
      [-105.56931736343999, 40.74088851285486],
      [-105.56925236342565, 40.74089351286025],
      [-105.

In [32]:
# return_count_only avoids pulling geometry/attributes when you only need a number
query_features(service_name, layer_id, where="acres > 100000", return_count_only=True)

{'count': 308}

In [33]:
# Spatial query: use a forest boundary as the AOI to clip another layer.
# Grab the Idaho Panhandle NF boundary...
ipnf = query_features(
    "EDW_ForestSystemBoundaries_01", 0,
    where="FORESTORGCODE='0104'",
)
aoi_geom = ipnf["features"][0]["geometry"]

# ...then query fires intersecting it (geometry can be a GeoJSON dict, Esri JSON, or a bbox string)
query_features(
    service_name, layer_id,
    geometry=aoi_geom, geometry_type="esriGeometryPolygon",
    out_fields="fire_name,ig_date", max_features=5,
)

{'type': 'FeatureCollection',
 'features': [],
 'exceededTransferLimit': True,
 'properties': {'exceededTransferLimit': True}}

**Known issue with the cell above:** `ipnf`'s geometry is a 161-part multipolygon with holes. `query_features` against it here returns 0 features, even though the fires genuinely intersect it — verified separately that `return_count_only=True` on the identical geometry correctly reports 40 matches, while asking the server to return the actual rows comes back with `exceededTransferLimit: true` and an empty feature list. This looks like an upstream ArcGIS Server limitation on highly complex query geometries, not a bug in `edw.py`'s geometry conversion — left in place for further testing.

A simpler AOI (fewer disjoint parts) works fine — e.g. a single ranger district boundary from the same forest:

In [34]:
# Priest Lake Ranger District boundary — same forest, but only 2 polygon parts
rd = query_features(
    "EDW_RangerDistricts_01", 0,
    where="districtname = 'Priest Lake Ranger District'",
)
rd_geom = rd["features"][0]["geometry"]

rd_fires = query_features(
    service_name, layer_id,
    geometry=rd_geom, geometry_type="esriGeometryPolygon",
    out_fields="fire_name,year,acres", max_features=5,
)
for f in rd_fires["features"]:
    p = f["properties"]
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

HUGHES 32 COMPLEX (2006): 4,533 acres
TOWER (2015): 24,761 acres
NORTH FORK HUGHES (2017): 4,624 acres
DIAMOND WATCH (2022): 1,432 acres
THOR (2022): 2,097 acres


In [35]:
# A plain bbox string also works as the geometry filter
bbox_fires = query_features(
    service_name, layer_id,
    geometry="-116.5,47.5,-116.0,48.0", geometry_type="esriGeometryEnvelope",
    out_fields="fire_name,year,acres", max_features=5,
)
for f in bbox_fires["features"]:
    p = f["properties"]
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

ULM PEAK (2006): 4,606 acres
CAPE HORN (2015): 1,505 acres
NORTH GRIZZLY (2015): 5,284 acres
WHITETAIL (2015): 2,022 acres
LOWER FLAT (2015): 9,023 acres


## 6. `query_features_with_pagination` — beyond the 2000-record server cap

Same signature as `query_features`, but `max_features` can exceed `_MAX_RECORD_COUNT`; it pages through `resultOffset` automatically. Using the point layer (fire ignition locations) here rather than the polygon layer — the EDW server 500s on exactly-2000-row pages of heavy polygon geometry, a server-side quirk unrelated to this function.

In [36]:
all_fires = query_features_with_pagination(
    service_name, 62,  # "Fire Occurrence Locations (All Years)" — points, not polygons
    out_fields="fire_name,ig_date,acres",
    max_features=4000,
)
len(all_fires["features"])

4000

## 7. `query_features_analytic` — SQL window functions

Runs ArcGIS's `queryAnalytic` operation (RANK, SUM, LAG/LEAD, PERCENTILE_CONT, ...). Unlike `query_features`, rows aren't collapsed — each analytic value is appended as a new field on its source feature.

Note: ArcGIS ignores whatever `out_name` you request for a `RANK` analytic and always names the computed field `rank_expr0` — filter on that name in `analytic_where`, not your requested `out_name`.

In [39]:
# Rank fires by acreage within each ignition year, keep only the #1 fire per year
biggest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{
        "type": "RANK",
        "field": "acres",
        "order_by": "acres DESC",
        "out_name": "acres_rank",
    }],
    partition_by="year",
    analytic_where="rank_expr0 = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
sorted(
    (f["properties"] for f in biggest_per_year["features"]),
    key=lambda p: p["year"],
)[-5:]

[{'fire_name': 'HERMITS PEAK',
  'year': 2022,
  'acres': 351782.0,
  'rank_expr0': 1},
 {'fire_name': 'YORK', 'year': 2023, 'acres': 94728.0, 'rank_expr0': 1},
 {'fire_name': 'SMOKEHOUSE CREEK',
  'year': 2024,
  'acres': 1047245.0,
  'rank_expr0': 1},
 {'fire_name': 'COTTONWOOD PEAK',
  'year': 2025,
  'acres': 137796.0,
  'rank_expr0': 1},
 {'fire_name': 'MORRILL', 'year': 2026, 'acres': 645316.0, 'rank_expr0': 1}]

## 8. `top_n_per_group` — convenience wrapper

Same query as above, without hand-rolling the RANK analytic. `descending=False` ranks smallest-first instead.

In [50]:
biggest_by_year = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=True,
    n=1,
    out_fields="fire_name,year,acres",
)

sorted(
    (f["properties"] for f in biggest_by_year["features"]),
    key=lambda p: p["year"],
)[-5:]

[{'fire_name': 'HERMITS PEAK',
  'year': 2022,
  'acres': 351782.0,
  'rank_expr0': 1},
 {'fire_name': 'YORK', 'year': 2023, 'acres': 94728.0, 'rank_expr0': 1},
 {'fire_name': 'SMOKEHOUSE CREEK',
  'year': 2024,
  'acres': 1047245.0,
  'rank_expr0': 1},
 {'fire_name': 'COTTONWOOD PEAK',
  'year': 2025,
  'acres': 137796.0,
  'rank_expr0': 1},
 {'fire_name': 'MORRILL', 'year': 2026, 'acres': 645316.0, 'rank_expr0': 1}]